<!-- project-navigation -->
# verification finale
Role: Model experiments and analysis.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [1]:
import pandas as pd

# ============================================================
# CHARGEMENT DES DATASETS FINAUX
# ============================================================

train_final = pd.read_csv(
    "data/processed/train_final.csv"
)

test_final = pd.read_csv(
    "data/processed/test_final.csv"
)

print("TRAIN :", train_final.shape)
print("TEST  :", test_final.shape)

print("\n==============================")
print("MISSING")
print("==============================")

print("Missing TRAIN :", train_final.isna().sum().sum())
print("Missing TEST  :", test_final.isna().sum().sum())

TRAIN : (307511, 290)
TEST  : (48744, 228)

MISSING
Missing TRAIN : 3966345
Missing TEST  : 0


In [2]:
# ============================================================
# COLONNES AVEC VALEURS MANQUANTES
# ============================================================

missing_train = (
    train_final.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_train = missing_train[missing_train > 0]

print("Nombre de colonnes contenant des NaN :", len(missing_train))

display(missing_train.head(30))

Nombre de colonnes contenant des NaN : 63


CC_NB_DELAY_MONTHS                    220606
CC_TOTAL_PAYMENT                      220606
CC_AVG_CREDIT_LIMIT                   220606
CC_MAX_BALANCE                        220606
CC_AVG_BALANCE                        220606
CC_NB_CREDIT_CARDS                    220606
CC_NB_CREDIT_CARD_RECORDS             220606
CC_AVG_PAYMENT                        220606
CC_DPD_RATIO                          220606
CC_MAX_DPD                            220606
CC_MAX_DPD_DEF                        220606
CC_CREDIT_UTILIZATION_RATIO           220606
BUREAU_BAL_NB_CREDITS_WITH_HISTORY     44020
BUREAU_BAL_AVG_MONTHS_HISTORY          44020
BUREAU_BAL_NB_CREDITS_WITH_DELAY       44020
BUREAU_BAL_MAX_DELAY                   44020
BUREAU_BAL_TOTAL_DELAY_MONTHS          44020
BUREAU_BAL_AVG_DELAY_RATIO             44020
BUREAU_BAL_TOTAL_OK_MONTHS             44020
BUREAU_NB_TOTAL_CREDITS                44020
BUREAU_NB_ACTIVE_CREDITS               44020
BUREAU_NB_OVERDUE_CREDITS              44020
BUREAU_TOT

In [3]:
# ============================================================
# COMPARAISON DES COLONNES TRAIN / TEST
# ============================================================

train_columns = set(train_final.columns)
test_columns = set(test_final.columns)

only_train = sorted(train_columns - test_columns)
only_test = sorted(test_columns - train_columns)

print("==========================================")
print("COLONNES TRAIN / TEST")
print("==========================================")

print("Nombre de colonnes TRAIN :", len(train_columns))
print("Nombre de colonnes TEST  :", len(test_columns))

print("\nColonnes présentes uniquement dans TRAIN :")
print("Nombre :", len(only_train))

for col in only_train:
    print("-", col)

print("\nColonnes présentes uniquement dans TEST :")
print("Nombre :", len(only_test))

for col in only_test:
    print("-", col)

COLONNES TRAIN / TEST
Nombre de colonnes TRAIN : 290
Nombre de colonnes TEST  : 228

Colonnes présentes uniquement dans TRAIN :
Nombre : 63
- BUREAU_AVG_CREDIT_AGE
- BUREAU_AVG_CREDIT_AMOUNT
- BUREAU_AVG_DEBT_RATIO
- BUREAU_BAL_AVG_DELAY_RATIO
- BUREAU_BAL_AVG_MONTHS_HISTORY
- BUREAU_BAL_MAX_DELAY
- BUREAU_BAL_NB_CREDITS_WITH_DELAY
- BUREAU_BAL_NB_CREDITS_WITH_HISTORY
- BUREAU_BAL_TOTAL_DELAY_MONTHS
- BUREAU_BAL_TOTAL_OK_MONTHS
- BUREAU_NB_ACTIVE_CREDITS
- BUREAU_NB_OVERDUE_CREDITS
- BUREAU_NB_TOTAL_CREDITS
- BUREAU_TOTAL_CREDIT
- BUREAU_TOTAL_CREDIT_LIMIT
- BUREAU_TOTAL_DEBT
- BUREAU_TOTAL_OVERDUE_AMOUNT
- CC_AVG_BALANCE
- CC_AVG_CREDIT_LIMIT
- CC_AVG_PAYMENT
- CC_CREDIT_UTILIZATION_RATIO
- CC_DPD_RATIO
- CC_MAX_BALANCE
- CC_MAX_DPD
- CC_MAX_DPD_DEF
- CC_NB_CREDIT_CARDS
- CC_NB_CREDIT_CARD_RECORDS
- CC_NB_DELAY_MONTHS
- CC_TOTAL_PAYMENT
- INST_AVG_PAYMENT_DELAY
- INST_AVG_PAYMENT_DIFF
- INST_LATE_PAYMENT_RATIO
- INST_MAX_PAYMENT_DELAY
- INST_NB_INSTALLMENTS
- INST_NB_LATE_PAYMENTS
- I

In [4]:
# ============================================================
# VÉRIFICATION DES IDS TRAIN / TEST
# ============================================================

# Charger les bases principales
train_base = pd.read_csv(
    "data/processed/application_train_encode.csv"
)

test_base = pd.read_csv(
    "data/processed/application_test_encode.csv"
)

# Charger les tables agrégées
bureau = pd.read_csv(
    "data/processed/bureau_features.csv"
)

bureau_balance = pd.read_csv(
    "data/processed/bureau_balance_features.csv"
)

installments = pd.read_csv(
    "data/processed/installments_features.csv"
)

previous = pd.read_csv(
    "data/processed/previous_application_features.csv"
)

pos_cash = pd.read_csv(
    "data/processed/pos_cash_features.csv"
)

credit_card = pd.read_csv(
    "data/processed/credit_card_features.csv"
)


# ============================================================
# TEST DE COUVERTURE DES IDS
# ============================================================

tables = {
    "BUREAU": bureau,
    "BUREAU_BALANCE": bureau_balance,
    "INSTALLMENTS": installments,
    "PREVIOUS": previous,
    "POS_CASH": pos_cash,
    "CREDIT_CARD": credit_card
}


for nom, table in tables.items():

    train_match = train_base["SK_ID_CURR"].isin(
        table["SK_ID_CURR"]
    ).sum()

    test_match = test_base["SK_ID_CURR"].isin(
        table["SK_ID_CURR"]
    ).sum()

    print("\n==========================================")
    print(nom)
    print("==========================================")

    print(
        "TRAIN avec historique :",
        train_match,
        "/",
        len(train_base)
    )

    print(
        "TRAIN sans historique :",
        len(train_base) - train_match
    )

    print(
        "TEST avec historique :",
        test_match,
        "/",
        len(test_base)
    )

    print(
        "TEST sans historique :",
        len(test_base) - test_match
    )


BUREAU
TRAIN avec historique : 263491 / 307511
TRAIN sans historique : 44020
TEST avec historique : 42320 / 48744
TEST sans historique : 6424

BUREAU_BALANCE
TRAIN avec historique : 263491 / 307511
TRAIN sans historique : 44020
TEST avec historique : 42320 / 48744
TEST sans historique : 6424

INSTALLMENTS
TRAIN avec historique : 291643 / 307511
TRAIN sans historique : 15868
TEST avec historique : 47944 / 48744
TEST sans historique : 800

PREVIOUS
TRAIN avec historique : 291057 / 307511
TRAIN sans historique : 16454
TEST avec historique : 47800 / 48744
TEST sans historique : 944

POS_CASH
TRAIN avec historique : 289444 / 307511
TRAIN sans historique : 18067
TEST avec historique : 47808 / 48744
TEST sans historique : 936

CREDIT_CARD
TRAIN avec historique : 86905 / 307511
TRAIN sans historique : 220606
TEST avec historique : 16653 / 48744
TEST sans historique : 32091


In [5]:
import pandas as pd

# ============================================================
# 1. CHARGEMENT DES BASES PRINCIPALES
# ============================================================

train_base = pd.read_csv(
    "data/processed/application_train_encode.csv"
)

test_base = pd.read_csv(
    "data/processed/application_test_encode.csv"
)

print("TRAIN BASE :", train_base.shape)
print("TEST BASE  :", test_base.shape)


# ============================================================
# 2. CHARGEMENT DES FEATURES AGRÉGÉES
# ============================================================

bureau = pd.read_csv(
    "data/processed/bureau_features.csv"
)

bureau_balance = pd.read_csv(
    "data/processed/bureau_balance_features.csv"
)

installments = pd.read_csv(
    "data/processed/installments_features.csv"
)

previous = pd.read_csv(
    "data/processed/previous_application_features.csv"
)

pos_cash = pd.read_csv(
    "data/processed/pos_cash_features.csv"
)

credit_card = pd.read_csv(
    "data/processed/credit_card_features.csv"
)


# ============================================================
# 3. DICTIONNAIRE DES TABLES
# ============================================================

tables_features = {
    "BUREAU": bureau,
    "BUREAU_BALANCE": bureau_balance,
    "INSTALLMENTS": installments,
    "PREVIOUS": previous,
    "POS_CASH": pos_cash,
    "CREDIT_CARD": credit_card
}


# ============================================================
# 4. COPIE DES BASES
# ============================================================

train_final = train_base.copy()
test_final = test_base.copy()


# ============================================================
# 5. MERGES IDENTIQUES SUR TRAIN ET TEST
# ============================================================

for nom, table in tables_features.items():

    train_final = train_final.merge(
        table,
        on="SK_ID_CURR",
        how="left"
    )

    test_final = test_final.merge(
        table,
        on="SK_ID_CURR",
        how="left"
    )

    print(
        f"{nom:20s} → "
        f"TRAIN {train_final.shape} | "
        f"TEST {test_final.shape}"
    )


# ============================================================
# 6. VÉRIFICATION APRÈS LES MERGES
# ============================================================

print("\n==========================================")
print("APRÈS TOUS LES MERGES")
print("==========================================")

print("TRAIN :", train_final.shape)
print("TEST  :", test_final.shape)

print(
    "NaN TRAIN :",
    train_final.isna().sum().sum()
)

print(
    "NaN TEST :",
    test_final.isna().sum().sum()
)

TRAIN BASE : (307511, 165)
TEST BASE  : (48744, 165)
BUREAU               → TRAIN (307511, 175) | TEST (48744, 175)
BUREAU_BALANCE       → TRAIN (307511, 182) | TEST (48744, 182)
INSTALLMENTS         → TRAIN (307511, 193) | TEST (48744, 193)
PREVIOUS             → TRAIN (307511, 205) | TEST (48744, 205)
POS_CASH             → TRAIN (307511, 216) | TEST (48744, 216)
CREDIT_CARD          → TRAIN (307511, 228) | TEST (48744, 228)

APRÈS TOUS LES MERGES
TRAIN : (307511, 228)
TEST  : (48744, 228)
NaN TRAIN : 3966345
NaN TEST : 524724


In [6]:
# ============================================================
# IDENTIFICATION DES FEATURES AGRÉGÉES
# ============================================================

# Les colonnes de la base initiale
base_columns = set(train_base.columns)

# Colonnes ajoutées par les tables agrégées
aggregated_columns = [
    col for col in train_final.columns
    if col not in base_columns
]

print("Nombre de features agrégées :", len(aggregated_columns))

print("\nFeatures agrégées :")

for col in aggregated_columns:
    print("-", col)

Nombre de features agrégées : 63

Features agrégées :
- NB_TOTAL_CREDITS
- NB_ACTIVE_CREDITS
- NB_OVERDUE_CREDITS
- TOTAL_CREDIT
- TOTAL_DEBT
- TOTAL_CREDIT_LIMIT
- TOTAL_OVERDUE_AMOUNT
- AVG_CREDIT_AMOUNT_x
- AVG_CREDIT_AGE
- AVG_DEBT_RATIO
- NB_CREDITS_WITH_HISTORY
- AVG_MONTHS_HISTORY
- MAX_DELAY
- TOTAL_DELAY_MONTHS
- AVG_DELAY_RATIO
- NB_CREDITS_WITH_DELAY
- TOTAL_OK_MONTHS
- NB_INSTALLMENTS
- NB_PREVIOUS_CREDITS
- TOTAL_INSTALLMENT_AMOUNT
- TOTAL_PAYMENT_AMOUNT
- NB_LATE_PAYMENTS
- AVG_PAYMENT_DELAY
- MAX_PAYMENT_DELAY
- NB_UNDERPAID
- AVG_PAYMENT_DIFF
- PAYMENT_RATIO
- LATE_PAYMENT_RATIO
- NB_PREVIOUS_APPLICATIONS
- AVG_APPLICATION_AMOUNT
- AVG_CREDIT_AMOUNT_y
- AVG_ANNUITY
- AVG_DOWN_PAYMENT
- AVG_CREDIT_DURATION
- NB_APPROVED
- NB_REFUSED
- NB_CANCELED
- NB_UNUSED
- APPROVAL_RATE
- REFUSAL_RATE
- NB_POS_RECORDS
- NB_POS_CREDITS
- NB_ACTIVE_POS
- NB_COMPLETED_POS
- NB_DELAY_MONTHS_x
- MAX_DPD_x
- MAX_DPD_DEF_x
- AVG_CNT_INSTALMENT
- AVG_CNT_INSTALMENT_FUTURE
- DPD_RATIO_x
- ACT

In [7]:
# ============================================================
# MISSING VALUES DANS LES FEATURES AGRÉGÉES
# ============================================================

missing_features = (
    train_final[aggregated_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("==========================================")
print("MISSING - FEATURES AGRÉGÉES")
print("==========================================")

display(
    missing_features[missing_features > 0]
)

MISSING - FEATURES AGRÉGÉES


MAX_DPD_DEF_y               220606
CREDIT_UTILIZATION_RATIO    220606
DPD_RATIO_y                 220606
MAX_BALANCE                 220606
AVG_BALANCE                 220606
                             ...  
PAYMENT_RATIO                15868
MAX_PAYMENT_DELAY            15868
TOTAL_PAYMENT_AMOUNT         15868
NB_LATE_PAYMENTS             15868
AVG_PAYMENT_DELAY            15868
Length: 63, dtype: int64

In [8]:
# ============================================================
# TRAITEMENT DES VALEURS MANQUANTES APRÈS LES MERGES
# ============================================================
#
# Les 165 variables de base ont déjà été traitées
# selon les règles définies précédemment :
#   - numérique → médiane
#   - catégorielle → mode
#   - absence d'événement → 0
#
# Les NaN actuels proviennent des LEFT JOIN avec les
# tables historiques.
#
# Ils signifient :
#   "le client n'a aucun historique dans cette source"
#
# → logique métier : remplacement par 0
# ============================================================


# Copie pour éviter toute modification inattendue
train_final = train_final.copy()
test_final = test_final.copy()


# ============================================================
# REMPLACEMENT DES NaN DES FEATURES AGRÉGÉES
# ============================================================

train_final[aggregated_columns] = (
    train_final[aggregated_columns].fillna(0)
)

test_final[aggregated_columns] = (
    test_final[aggregated_columns].fillna(0)
)


# ============================================================
# VÉRIFICATION
# ============================================================

print("==========================================")
print("APRÈS TRAITEMENT DES FEATURES AGRÉGÉES")
print("==========================================")

print(
    "Valeurs manquantes TRAIN :",
    train_final.isna().sum().sum()
)

print(
    "Valeurs manquantes TEST :",
    test_final.isna().sum().sum()
)

APRÈS TRAITEMENT DES FEATURES AGRÉGÉES
Valeurs manquantes TRAIN : 0
Valeurs manquantes TEST : 0


In [9]:
# ============================================================
# VÉRIFICATION FINALE TRAIN / TEST
# ============================================================

train_columns = set(train_final.columns)
test_columns = set(test_final.columns)

only_train = sorted(train_columns - test_columns)
only_test = sorted(test_columns - train_columns)

print("==========================================")
print("ALIGNEMENT TRAIN / TEST")
print("==========================================")

print("Nombre de colonnes TRAIN :", len(train_columns))
print("Nombre de colonnes TEST  :", len(test_columns))

print("\nColonnes uniquement dans TRAIN :")
print(only_train)

print("\nColonnes uniquement dans TEST :")
print(only_test)

ALIGNEMENT TRAIN / TEST
Nombre de colonnes TRAIN : 228
Nombre de colonnes TEST  : 228

Colonnes uniquement dans TRAIN :
[]

Colonnes uniquement dans TEST :
[]


In [10]:
# ============================================================
# SAUVEGARDE DES DATASETS FINAUX
# ============================================================

train_final.to_csv(
    "data/processed/train_final.csv",
    index=False
)

test_final.to_csv(
    "data/processed/test_final.csv",
    index=False
)

print("==========================================")
print("FICHIERS FINAUX SAUVEGARDÉS")
print("==========================================")

print("TRAIN :", train_final.shape)
print("TEST  :", test_final.shape)

print(
    "NaN TRAIN :",
    train_final.isna().sum().sum()
)

print(
    "NaN TEST :",
    test_final.isna().sum().sum()
)

FICHIERS FINAUX SAUVEGARDÉS
TRAIN : (307511, 228)
TEST  : (48744, 228)
NaN TRAIN : 0
NaN TEST : 0


In [11]:
# ============================================================
# VÉRIFICATION APRÈS RECHARGEMENT
# ============================================================

train_check = pd.read_csv(
    "data/processed/train_final.csv"
)

test_check = pd.read_csv(
    "data/processed/test_final.csv"
)

print("==========================================")
print("VÉRIFICATION DES FICHIERS SAUVEGARDÉS")
print("==========================================")

print("TRAIN :", train_check.shape)
print("TEST  :", test_check.shape)

print(
    "NaN TRAIN :",
    train_check.isna().sum().sum()
)

print(
    "NaN TEST :",
    test_check.isna().sum().sum()
)

print(
    "Colonnes identiques :",
    list(train_check.columns) == list(test_check.columns)
)

VÉRIFICATION DES FICHIERS SAUVEGARDÉS
TRAIN : (307511, 228)
TEST  : (48744, 228)
NaN TRAIN : 0
NaN TEST : 0
Colonnes identiques : True


In [15]:
import pandas as pd
import numpy as np

# ============================================================
# CHARGEMENT DES DATASETS FINAUX
# ============================================================

train_final = pd.read_csv(
    "data/processed/train_final.csv"
)

test_final = pd.read_csv(
    "data/processed/test_final.csv"
)

print("==========================================")
print("CHARGEMENT")
print("==========================================")

print("TRAIN :", train_final.shape)
print("TEST  :", test_final.shape)

print("NaN TRAIN :", train_final.isna().sum().sum())
print("NaN TEST  :", test_final.isna().sum().sum())

CHARGEMENT
TRAIN : (307511, 228)
TEST  : (48744, 228)
NaN TRAIN : 0
NaN TEST  : 0


In [16]:
# ============================================================
# VARIABLES CATÉGORIELLES
# ============================================================

categorical_columns = [
    "HOUSETYPE_MODE",
    "WALLSMATERIAL_MODE",
    "age_tranche"
]

print("Variables catégorielles :")
for col in categorical_columns:
    print("-", col)

# Vérification
print("\nTypes :")
print(train_final[categorical_columns].dtypes)

Variables catégorielles :
- HOUSETYPE_MODE
- WALLSMATERIAL_MODE
- age_tranche

Types :
HOUSETYPE_MODE        str
WALLSMATERIAL_MODE    str
age_tranche           str
dtype: object


In [17]:
# ============================================================
# VÉRIFICATION DES CATÉGORIES TRAIN / TEST
# ============================================================

for col in categorical_columns:

    print("\n==========================================")
    print(col)
    print("==========================================")

    print("TRAIN :")
    print(train_final[col].value_counts())

    print("\nTEST :")
    print(test_final[col].value_counts())


HOUSETYPE_MODE
TRAIN :
HOUSETYPE_MODE
Inconnu             154297
block of flats      150503
specific housing      1499
terraced house        1212
Name: count, dtype: int64

TEST :
HOUSETYPE_MODE
0    48744
Name: count, dtype: int64

WALLSMATERIAL_MODE
TRAIN :
WALLSMATERIAL_MODE
Inconnu         156341
Panel            66040
Stone, brick     64815
Block             9253
Wooden            5362
Mixed             2296
Monolithic        1779
Others            1625
Name: count, dtype: int64

TEST :
WALLSMATERIAL_MODE
0    48744
Name: count, dtype: int64

age_tranche
TRAIN :
age_tranche
30-40    83117
40-50    74401
50-60    67819
20-30    52806
60-70    29368
Name: count, dtype: int64

TEST :
age_tranche
30-40    13265
40-50    12165
50-60    10521
20-30     8050
60-70     4743
Name: count, dtype: int64


In [18]:
from sklearn.preprocessing import OneHotEncoder

# ============================================================
# ONE-HOT ENCODING
# ============================================================

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apprentissage UNIQUEMENT sur TRAIN
encoder.fit(train_final[categorical_columns])

print("Encodage terminé.")
print("Nombre de nouvelles variables :",
      len(encoder.get_feature_names_out(categorical_columns)))

Encodage terminé.
Nombre de nouvelles variables : 17


In [20]:
# ============================================================
# SUPPRESSION DES VARIABLES CATÉGORIELLES ORIGINALES
# ============================================================

train_numeric = train_final.drop(
    columns=categorical_columns
)

test_numeric = test_final.drop(
    columns=categorical_columns
)

print("TRAIN sans variables catégorielles :", train_numeric.shape)
print("TEST sans variables catégorielles  :", test_numeric.shape)

TRAIN sans variables catégorielles : (307511, 225)
TEST sans variables catégorielles  : (48744, 225)


In [23]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder

# ============================================================
# 1. RECHARGEMENT PROPRE
# ============================================================

train_final = pd.read_csv(
    "data/processed/train_final.csv"
)

test_final = pd.read_csv(
    "data/processed/test_final.csv"
)

# ============================================================
# 2. VARIABLES CATÉGORIELLES
# ============================================================

categorical_columns = [
    "HOUSETYPE_MODE",
    "WALLSMATERIAL_MODE",
    "age_tranche"
]

# ============================================================
# 3. CONVERSION EXPLICITE EN OBJECT
# ============================================================

for col in categorical_columns:
    train_final[col] = train_final[col].astype(object)
    test_final[col] = test_final[col].astype(object)

print("==========================================")
print("TYPES DES VARIABLES")
print("==========================================")

print(train_final[categorical_columns].dtypes)
print()
print(test_final[categorical_columns].dtypes)

# ============================================================
# 4. CRÉATION DE L'ENCODEUR
# ============================================================

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# IMPORTANT :
# apprentissage uniquement sur TRAIN
encoder.fit(
    train_final[categorical_columns]
)

# ============================================================
# 5. TRANSFORMATION
# ============================================================

train_encoded_array = encoder.transform(
    train_final[categorical_columns]
)

test_encoded_array = encoder.transform(
    test_final[categorical_columns]
)

print("\n==========================================")
print("TRANSFORMATION TERMINÉE")
print("==========================================")

print(
    "TRAIN array :",
    train_encoded_array.shape
)

print(
    "TEST array :",
    test_encoded_array.shape
)

TYPES DES VARIABLES
HOUSETYPE_MODE        object
WALLSMATERIAL_MODE    object
age_tranche           object
dtype: object

HOUSETYPE_MODE        object
WALLSMATERIAL_MODE    object
age_tranche           object
dtype: object

TRANSFORMATION TERMINÉE
TRAIN array : (307511, 17)
TEST array : (48744, 17)


In [24]:
# ============================================================
# 6. DATAFRAMES ENCODÉS
# ============================================================

encoded_columns = encoder.get_feature_names_out(
    categorical_columns
)

train_encoded_cat = pd.DataFrame(
    train_encoded_array,
    columns=encoded_columns,
    index=train_final.index
)

test_encoded_cat = pd.DataFrame(
    test_encoded_array,
    columns=encoded_columns,
    index=test_final.index
)

print("==========================================")
print("DATAFRAMES ENCODÉS")
print("==========================================")

print("TRAIN :", train_encoded_cat.shape)
print("TEST  :", test_encoded_cat.shape)

print("\nColonnes créées :")
print(encoded_columns.tolist())

DATAFRAMES ENCODÉS
TRAIN : (307511, 17)
TEST  : (48744, 17)

Colonnes créées :
['HOUSETYPE_MODE_Inconnu', 'HOUSETYPE_MODE_block of flats', 'HOUSETYPE_MODE_specific housing', 'HOUSETYPE_MODE_terraced house', 'WALLSMATERIAL_MODE_Block', 'WALLSMATERIAL_MODE_Inconnu', 'WALLSMATERIAL_MODE_Mixed', 'WALLSMATERIAL_MODE_Monolithic', 'WALLSMATERIAL_MODE_Others', 'WALLSMATERIAL_MODE_Panel', 'WALLSMATERIAL_MODE_Stone, brick', 'WALLSMATERIAL_MODE_Wooden', 'age_tranche_20-30', 'age_tranche_30-40', 'age_tranche_40-50', 'age_tranche_50-60', 'age_tranche_60-70']


In [25]:
X = train_final.drop(
    columns=["TARGET", "SK_ID_CURR"],
    errors="ignore"
)

y = train_final["TARGET"]

print("X :", X.shape)
print("y :", y.shape)

X : (307511, 226)
y : (307511,)


In [26]:
# ============================================================
# 7. SUPPRESSION DES 3 VARIABLES ORIGINALES
# ============================================================

train_numeric = train_final.drop(
    columns=categorical_columns
)

test_numeric = test_final.drop(
    columns=categorical_columns
)

# ============================================================
# 8. CONCATÉNATION
# ============================================================

train_encoded = pd.concat(
    [train_numeric, train_encoded_cat],
    axis=1
)

test_encoded = pd.concat(
    [test_numeric, test_encoded_cat],
    axis=1
)

print("==========================================")
print("DATASETS ENTIÈREMENT ENCODÉS")
print("==========================================")

print("TRAIN :", train_encoded.shape)
print("TEST  :", test_encoded.shape)

DATASETS ENTIÈREMENT ENCODÉS
TRAIN : (307511, 242)
TEST  : (48744, 242)


In [27]:
# ============================================================
# 9. CONTRÔLE FINAL
# ============================================================

print("==========================================")
print("CONTRÔLE FINAL")
print("==========================================")

print("NaN TRAIN :",
      train_encoded.isna().sum().sum())

print("NaN TEST :",
      test_encoded.isna().sum().sum())

print(
    "Colonnes identiques :",
    list(train_encoded.columns) == list(test_encoded.columns)
)

print(
    "Variables non numériques TRAIN :",
    train_encoded.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

print(
    "Variables non numériques TEST :",
    test_encoded.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

CONTRÔLE FINAL
NaN TRAIN : 0
NaN TEST : 0
Colonnes identiques : True
Variables non numériques TRAIN : []
Variables non numériques TEST : []


In [28]:
# ============================================================
# 10. SAUVEGARDE DES DATASETS ENCODÉS
# ============================================================

train_encoded.to_csv(
    "data/processed/train_encoded.csv",
    index=False
)

test_encoded.to_csv(
    "data/processed/test_encoded.csv",
    index=False
)

print("==========================================")
print("SAUVEGARDE TERMINÉE")
print("==========================================")

print("TRAIN sauvegardé :", train_encoded.shape)
print("TEST sauvegardé  :", test_encoded.shape)

print(
    "NaN TRAIN :",
    train_encoded.isna().sum().sum()
)

print(
    "NaN TEST :",
    test_encoded.isna().sum().sum()
)

SAUVEGARDE TERMINÉE
TRAIN sauvegardé : (307511, 242)
TEST sauvegardé  : (48744, 242)
NaN TRAIN : 0
NaN TEST : 0


In [29]:
# ============================================================
# 11. VÉRIFICATION APRÈS RECHARGEMENT
# ============================================================

train_check = pd.read_csv(
    "data/processed/train_encoded.csv"
)

test_check = pd.read_csv(
    "data/processed/test_encoded.csv"
)

print("==========================================")
print("VÉRIFICATION DES FICHIERS")
print("==========================================")

print("TRAIN :", train_check.shape)
print("TEST  :", test_check.shape)

print("NaN TRAIN :", train_check.isna().sum().sum())
print("NaN TEST  :", test_check.isna().sum().sum())

print(
    "Colonnes identiques :",
    list(train_check.columns) == list(test_check.columns)
)

print(
    "TRAIN entièrement numérique :",
    len(train_check.select_dtypes(exclude=np.number).columns) == 0
)

print(
    "TEST entièrement numérique :",
    len(test_check.select_dtypes(exclude=np.number).columns) == 0
)

VÉRIFICATION DES FICHIERS
TRAIN : (307511, 242)
TEST  : (48744, 242)
NaN TRAIN : 0
NaN TEST  : 0
Colonnes identiques : True
TRAIN entièrement numérique : True
TEST entièrement numérique : True


In [30]:
# ============================================================
# AFFICHAGE DE TOUTES LES FEATURES APRÈS ENCODAGE
# ============================================================

print("==========================================")
print("TOUTES LES FEATURES")
print("==========================================")

print("Nombre total de features TRAIN :", train_encoded.shape[1])
print("Nombre total de features TEST  :", test_encoded.shape[1])

print("\nListe complète des features :\n")

for i, col in enumerate(train_encoded.columns, start=1):
    print(f"{i:3d}. {col}")

TOUTES LES FEATURES
Nombre total de features TRAIN : 242
Nombre total de features TEST  : 242

Liste complète des features :

  1. SK_ID_CURR
  2. TARGET
  3. NAME_CONTRACT_TYPE
  4. CODE_GENDER
  5. FLAG_OWN_CAR
  6. FLAG_OWN_REALTY
  7. CNT_CHILDREN
  8. AMT_INCOME_TOTAL
  9. AMT_CREDIT
 10. AMT_ANNUITY
 11. AMT_GOODS_PRICE
 12. REGION_POPULATION_RELATIVE
 13. DAYS_BIRTH
 14. DAYS_EMPLOYED
 15. DAYS_REGISTRATION
 16. DAYS_ID_PUBLISH
 17. OWN_CAR_AGE
 18. FLAG_EMP_PHONE
 19. FLAG_WORK_PHONE
 20. FLAG_CONT_MOBILE
 21. FLAG_PHONE
 22. FLAG_EMAIL
 23. CNT_FAM_MEMBERS
 24. REGION_RATING_CLIENT
 25. REGION_RATING_CLIENT_W_CITY
 26. HOUR_APPR_PROCESS_START
 27. REG_REGION_NOT_LIVE_REGION
 28. REG_REGION_NOT_WORK_REGION
 29. LIVE_REGION_NOT_WORK_REGION
 30. REG_CITY_NOT_LIVE_CITY
 31. REG_CITY_NOT_WORK_CITY
 32. LIVE_CITY_NOT_WORK_CITY
 33. EXT_SOURCE_1
 34. EXT_SOURCE_2
 35. EXT_SOURCE_3
 36. OBS_30_CNT_SOCIAL_CIRCLE
 37. DEF_30_CNT_SOCIAL_CIRCLE
 38. OBS_60_CNT_SOCIAL_CIRCLE
 39. DEF_60_CN

In [31]:
# ============================================================
# FEATURES + TYPES
# ============================================================

features_info = pd.DataFrame({
    "Feature": train_encoded.columns,
    "Type": train_encoded.dtypes.astype(str).values
})

display(features_info)

,Feature,Type
0,SK_ID_CURR,int64
1,TARGET,int64
2,NAME_CONTRACT_TYPE,int64
3,CODE_GENDER,int64
4,FLAG_OWN_CAR,int64
...,...,...
237,age_tranche_20-30,float64
238,age_tranche_30-40,float64
239,age_tranche_40-50,float64
240,age_tranche_50-60,float64


In [32]:
# ============================================================
# APERÇU DE TOUTES LES FEATURES
# ============================================================

pd.set_option("display.max_columns", None)

display(
    train_encoded.head()
)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,age_years,OWN_CAR_AGE_MISSING,DAYS_EMPLOYED_ANOM,AMT_INCOME_TOTAL_LOG,NAME_TYPE_SUITE_Family,NAME_TYPE_SUITE_Group of people,NAME_TYPE_SUITE_Other_A,NAME_TYPE_SUITE_Other_B,"NAME_TYPE_SUITE_Spouse, partner",NAME_TYPE_SUITE_Unaccompanied,NAME_INCOME_TYPE_Commercial associate,NAME_INCOME_TYPE_Maternity leave,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Unemployed,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Unknown,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,OCCUPATION_TYPE_Waiters/barmen staff,WEEKDAY_APPR_PROCESS_START_MONDAY,WEEKDAY_APPR_PROCESS_START_SATURDAY,WEEKDAY_APPR_PROCESS_START_SUNDAY,WEEKDAY_APPR_PROCESS_START_THURSDAY,WEEKDAY_APPR_PROCESS_START_TUESDAY,WEEKDAY_APPR_PROCESS_START_WEDNESDAY,ORGANIZATION_TYPE_Agriculture,ORGANIZATION_TYPE_Bank,ORGANIZATION_TYPE_Business Entity Type 1,ORGANIZATION_TYPE_Business Entity Type 2,ORGANIZATION_TYPE_Business Entity Type 3,ORGANIZATION_TYPE_Cleaning,ORGANIZATION_TYPE_Construction,ORGANIZATION_TYPE_Culture,ORGANIZATION_TYPE_Electricity,ORGANIZATION_TYPE_Emergency,ORGANIZATION_TYPE_Government,ORGANIZATION_TYPE_Hotel,ORGANIZATION_TYPE_Housing,ORGANIZATION_TYPE_Industry: type 1,ORGANIZATION_TYPE_Industry: type 10,ORGANIZATION_TYPE_Industry: type 11,ORGANIZATION_TYPE_Industry: type 12,ORGANIZATION_TYPE_Industry: type 13,ORGANIZATION_TYPE_Industry: type 2,ORGANIZATION_TYPE_Industry: type 3,ORGANIZATION_TYPE_Industry: type 4,ORGANIZATION_TYPE_Industry: type 5,ORGANIZATION_TYPE_Industry: type 6,ORGANIZATION_TYPE_Industry: type 7,ORGANIZATION_TYPE_Industry: type 8,ORGANIZATION_TYPE_Industry: type 9,ORGANIZATION_TYPE_Insurance,ORGANIZATION_TYPE_Kindergarten,ORGANIZATION_TYPE_Legal Services,ORGANIZATION_TYPE_Medicine,ORGANIZATION_TYPE_Military,ORGANIZATION_TYPE_Mobile,ORGANIZATION_TYPE_Other,ORGANIZATION_TYPE_Police,ORGANIZATION_TYPE_Postal,ORGANIZATION_TYPE_Realtor,ORGANIZATION_TYPE_Religion,ORGANIZATION_TYPE_Restaurant,ORGANIZATION_TYPE_School,ORGANIZATION_TYPE_Security,ORGANIZATION_TYPE_Security Ministries,ORGANIZATION_TYPE_Self-employed,ORGANIZATION_TYPE_Services,ORGANIZATION_TYPE_Telecom,ORGANIZATION_TYPE_Trade: typ

In [33]:
print("Nombre total de colonnes TRAIN :", train_encoded.shape[1])
print("Nombre de features :", train_encoded.shape[1] - 1)  # - TARGET

Nombre total de colonnes TRAIN : 242
Nombre de features : 241


In [35]:
print("==========================================")
print("BILAN DES FEATURES")
print("==========================================")

print("Features avant agrégation :", train_base.shape[1])
print("Features après agrégation :", train_final.shape[1])
print("Features après encodage   :", train_encoded.shape[1])

print("\nFeatures agrégées ajoutées :", 
      train_final.shape[1] - train_base.shape[1])

print("Features ajoutées nettes par encodage :",
      train_encoded.shape[1] - train_final.shape[1])

print("\nNombre total de features finales :",
      train_encoded.shape[1])

BILAN DES FEATURES
Features avant agrégation : 165
Features après agrégation : 228
Features après encodage   : 242

Features agrégées ajoutées : 63
Features ajoutées nettes par encodage : 14

Nombre total de features finales : 242


In [38]:
# ============================================================
# FEATURES AJOUTÉES AU TOTAL PAR RAPPORT À LA BASE INITIALE
# ============================================================

base_columns = set(train_base.columns)

new_features = [
    col for col in train_encoded.columns
    if col not in base_columns
]

print("==========================================")
print("FEATURES AJOUTÉES AU TOTAL")
print("==========================================")

print("Nombre total :", len(new_features))

for i, col in enumerate(new_features, 1):
    print(f"{i:02d}. {col}")

FEATURES AJOUTÉES AU TOTAL
Nombre total : 80
01. NB_TOTAL_CREDITS
02. NB_ACTIVE_CREDITS
03. NB_OVERDUE_CREDITS
04. TOTAL_CREDIT
05. TOTAL_DEBT
06. TOTAL_CREDIT_LIMIT
07. TOTAL_OVERDUE_AMOUNT
08. AVG_CREDIT_AMOUNT_x
09. AVG_CREDIT_AGE
10. AVG_DEBT_RATIO
11. NB_CREDITS_WITH_HISTORY
12. AVG_MONTHS_HISTORY
13. MAX_DELAY
14. TOTAL_DELAY_MONTHS
15. AVG_DELAY_RATIO
16. NB_CREDITS_WITH_DELAY
17. TOTAL_OK_MONTHS
18. NB_INSTALLMENTS
19. NB_PREVIOUS_CREDITS
20. TOTAL_INSTALLMENT_AMOUNT
21. TOTAL_PAYMENT_AMOUNT
22. NB_LATE_PAYMENTS
23. AVG_PAYMENT_DELAY
24. MAX_PAYMENT_DELAY
25. NB_UNDERPAID
26. AVG_PAYMENT_DIFF
27. PAYMENT_RATIO
28. LATE_PAYMENT_RATIO
29. NB_PREVIOUS_APPLICATIONS
30. AVG_APPLICATION_AMOUNT
31. AVG_CREDIT_AMOUNT_y
32. AVG_ANNUITY
33. AVG_DOWN_PAYMENT
34. AVG_CREDIT_DURATION
35. NB_APPROVED
36. NB_REFUSED
37. NB_CANCELED
38. NB_UNUSED
39. APPROVAL_RATE
40. REFUSAL_RATE
41. NB_POS_RECORDS
42. NB_POS_CREDITS
43. NB_ACTIVE_POS
44. NB_COMPLETED_POS
45. NB_DELAY_MONTHS_x
46. MAX_DPD_x
4

In [39]:
import pandas as pd
import os

# ============================================================
# 1. LISTE DES 63 FEATURES AGRÉGÉES
# ============================================================

aggregated_features = [
    "NB_TOTAL_CREDITS",
    "NB_ACTIVE_CREDITS",
    "NB_OVERDUE_CREDITS",
    "TOTAL_CREDIT",
    "TOTAL_DEBT",
    "TOTAL_CREDIT_LIMIT",
    "TOTAL_OVERDUE_AMOUNT",
    "AVG_CREDIT_AMOUNT_x",
    "AVG_CREDIT_AGE",
    "AVG_DEBT_RATIO",
    "NB_CREDITS_WITH_HISTORY",
    "AVG_MONTHS_HISTORY",
    "MAX_DELAY",
    "TOTAL_DELAY_MONTHS",
    "AVG_DELAY_RATIO",
    "NB_CREDITS_WITH_DELAY",
    "TOTAL_OK_MONTHS",
    "NB_INSTALLMENTS",
    "NB_PREVIOUS_CREDITS",
    "TOTAL_INSTALLMENT_AMOUNT",
    "TOTAL_PAYMENT_AMOUNT",
    "NB_LATE_PAYMENTS",
    "AVG_PAYMENT_DELAY",
    "MAX_PAYMENT_DELAY",
    "NB_UNDERPAID",
    "AVG_PAYMENT_DIFF",
    "PAYMENT_RATIO",
    "LATE_PAYMENT_RATIO",
    "NB_PREVIOUS_APPLICATIONS",
    "AVG_APPLICATION_AMOUNT",
    "AVG_CREDIT_AMOUNT_y",
    "AVG_ANNUITY",
    "AVG_DOWN_PAYMENT",
    "AVG_CREDIT_DURATION",
    "NB_APPROVED",
    "NB_REFUSED",
    "NB_CANCELED",
    "NB_UNUSED",
    "APPROVAL_RATE",
    "REFUSAL_RATE",
    "NB_POS_RECORDS",
    "NB_POS_CREDITS",
    "NB_ACTIVE_POS",
    "NB_COMPLETED_POS",
    "NB_DELAY_MONTHS_x",
    "MAX_DPD_x",
    "MAX_DPD_DEF_x",
    "AVG_CNT_INSTALMENT",
    "AVG_CNT_INSTALMENT_FUTURE",
    "DPD_RATIO_x",
    "ACTIVE_RATIO",
    "NB_CREDIT_CARD_RECORDS",
    "NB_CREDIT_CARDS",
    "AVG_BALANCE",
    "MAX_BALANCE",
    "AVG_CREDIT_LIMIT",
    "AVG_PAYMENT",
    "TOTAL_PAYMENT",
    "NB_DELAY_MONTHS_y",
    "MAX_DPD_y",
    "MAX_DPD_DEF_y",
    "CREDIT_UTILIZATION_RATIO",
    "DPD_RATIO_y"
]


# ============================================================
# 2. FEATURES CRÉÉES PAR ENCODAGE ONE-HOT
# ============================================================

encoded_features = [
    "HOUSETYPE_MODE_Inconnu",
    "HOUSETYPE_MODE_block of flats",
    "HOUSETYPE_MODE_specific housing",
    "HOUSETYPE_MODE_terraced house",
    "WALLSMATERIAL_MODE_Block",
    "WALLSMATERIAL_MODE_Inconnu",
    "WALLSMATERIAL_MODE_Mixed",
    "WALLSMATERIAL_MODE_Monolithic",
    "WALLSMATERIAL_MODE_Others",
    "WALLSMATERIAL_MODE_Panel",
    "WALLSMATERIAL_MODE_Stone, brick",
    "WALLSMATERIAL_MODE_Wooden",
    "age_tranche_20-30",
    "age_tranche_30-40",
    "age_tranche_40-50",
    "age_tranche_50-60",
    "age_tranche_60-70"
]


# ============================================================
# 3. INFORMATIONS SUR LES FEATURES AGRÉGÉES
# ============================================================

feature_info = {

# ---------------- BUREAU ----------------

"NB_TOTAL_CREDITS": (
    "BUREAU",
    "Comptage",
    "Nombre total de crédits présents dans l'historique externe",
    "Mesure l'expérience de crédit du client. Un nombre élevé indique un historique financier plus important."
),

"NB_ACTIVE_CREDITS": (
    "BUREAU",
    "Comptage",
    "Nombre de crédits encore actifs",
    "Permet d'évaluer les engagements financiers actuellement en cours."
),

"NB_OVERDUE_CREDITS": (
    "BUREAU",
    "Comptage",
    "Nombre de crédits en retard",
    "Indicateur direct de difficultés de remboursement et donc du risque de défaut."
),

"TOTAL_CREDIT": (
    "BUREAU",
    "Somme",
    "Montant total des crédits externes",
    "Mesure l'exposition globale du client au crédit."
),

"TOTAL_DEBT": (
    "BUREAU",
    "Somme",
    "Dette totale associée aux crédits externes",
    "Permet d'évaluer le niveau d'endettement du client."
),

"TOTAL_CREDIT_LIMIT": (
    "BUREAU",
    "Somme",
    "Limite totale de crédit disponible",
    "Mesure la capacité de crédit totale accordée au client."
),

"TOTAL_OVERDUE_AMOUNT": (
    "BUREAU",
    "Somme",
    "Montant total des sommes en retard",
    "Un montant élevé peut indiquer des difficultés financières."
),

"AVG_CREDIT_AMOUNT_x": (
    "BUREAU",
    "Moyenne",
    "Montant moyen des crédits externes",
    "Permet de caractériser la taille habituelle des crédits du client."
),

"AVG_CREDIT_AGE": (
    "BUREAU",
    "Moyenne",
    "Âge moyen des crédits externes",
    "Indique l'ancienneté moyenne de l'historique de crédit."
),

"AVG_DEBT_RATIO": (
    "BUREAU",
    "Moyenne",
    "Ratio moyen d'endettement",
    "Mesure la proportion de dette par rapport aux montants de crédit."
),

"NB_CREDITS_WITH_HISTORY": (
    "BUREAU_BALANCE",
    "Comptage",
    "Nombre de crédits possédant un historique mensuel",
    "Indique la quantité de crédits pour lesquels un suivi historique est disponible."
),

"AVG_MONTHS_HISTORY": (
    "BUREAU_BALANCE",
    "Moyenne",
    "Nombre moyen de mois d'historique",
    "Mesure la profondeur de l'historique financier disponible."
),

"MAX_DELAY": (
    "BUREAU_BALANCE",
    "Maximum",
    "Retard maximal observé",
    "Permet d'identifier les retards de remboursement les plus importants."
),

"TOTAL_DELAY_MONTHS": (
    "BUREAU_BALANCE",
    "Somme",
    "Nombre total de mois avec retard",
    "Mesure la fréquence globale des retards."
),

"AVG_DELAY_RATIO": (
    "BUREAU_BALANCE",
    "Moyenne",
    "Ratio moyen de retard",
    "Mesure la proportion de périodes présentant un retard."
),

"NB_CREDITS_WITH_DELAY": (
    "BUREAU_BALANCE",
    "Comptage",
    "Nombre de crédits ayant connu un retard",
    "Indicateur du comportement de remboursement passé."
),

"TOTAL_OK_MONTHS": (
    "BUREAU_BALANCE",
    "Somme",
    "Nombre total de mois sans retard",
    "Mesure la régularité des remboursements."
),

# ---------------- INSTALLMENTS ----------------

"NB_INSTALLMENTS": (
    "INSTALLMENTS",
    "Comptage",
    "Nombre total d'échéances",
    "Mesure le volume d'échéances observées dans l'historique."
),

"NB_PREVIOUS_CREDITS": (
    "INSTALLMENTS",
    "Comptage",
    "Nombre de crédits précédents concernés",
    "Indique l'expérience passée du client en matière de remboursement."
),

"TOTAL_INSTALLMENT_AMOUNT": (
    "INSTALLMENTS",
    "Somme",
    "Montant total prévu des échéances",
    "Mesure le volume financier total des échéances."
),

"TOTAL_PAYMENT_AMOUNT": (
    "INSTALLMENTS",
    "Somme",
    "Montant total effectivement payé",
    "Permet de comparer les paiements réalisés aux paiements attendus."
),

"NB_LATE_PAYMENTS": (
    "INSTALLMENTS",
    "Comptage",
    "Nombre de paiements effectués en retard",
    "Indicateur important du comportement de remboursement."
),

"AVG_PAYMENT_DELAY": (
    "INSTALLMENTS",
    "Moyenne",
    "Retard moyen de paiement",
    "Mesure la durée moyenne des retards."
),

"MAX_PAYMENT_DELAY": (
    "INSTALLMENTS",
    "Maximum",
    "Retard maximal de paiement",
    "Permet d'identifier les retards exceptionnels."
),

"NB_UNDERPAID": (
    "INSTALLMENTS",
    "Comptage",
    "Nombre d'échéances sous-payées",
    "Indique les situations où le client n'a pas payé le montant attendu."
),

"AVG_PAYMENT_DIFF": (
    "INSTALLMENTS",
    "Moyenne",
    "Différence moyenne entre paiement et montant prévu",
    "Mesure l'écart moyen entre ce qui devait être payé et ce qui a réellement été payé."
),

"PAYMENT_RATIO": (
    "INSTALLMENTS",
    "Ratio",
    "Ratio entre montant payé et montant des échéances",
    "Mesure le niveau de réalisation des remboursements."
),

"LATE_PAYMENT_RATIO": (
    "INSTALLMENTS",
    "Ratio",
    "Proportion de paiements effectués en retard",
    "Permet de comparer le comportement de remboursement entre clients ayant des volumes d'échéances différents."
),

# ---------------- PREVIOUS APPLICATION ----------------

"NB_PREVIOUS_APPLICATIONS": (
    "PREVIOUS",
    "Comptage",
    "Nombre total de demandes de crédit précédentes",
    "Mesure l'historique des demandes de crédit du client."
),

"AVG_APPLICATION_AMOUNT": (
    "PREVIOUS",
    "Moyenne",
    "Montant moyen demandé",
    "Indique le niveau de financement généralement recherché."
),

"AVG_CREDIT_AMOUNT_y": (
    "PREVIOUS",
    "Moyenne",
    "Montant moyen de crédit accordé",
    "Mesure le niveau moyen de financement obtenu."
),

"AVG_ANNUITY": (
    "PREVIOUS",
    "Moyenne",
    "Annuité moyenne des crédits précédents",
    "Permet d'évaluer le niveau moyen des remboursements demandés."
),

"AVG_DOWN_PAYMENT": (
    "PREVIOUS",
    "Moyenne",
    "Apport initial moyen",
    "Indique la capacité du client à financer une partie du projet initialement."
),

"AVG_CREDIT_DURATION": (
    "PREVIOUS",
    "Moyenne",
    "Durée moyenne des crédits précédents",
    "Caractérise la structure temporelle des engagements passés."
),

"NB_APPROVED": (
    "PREVIOUS",
    "Comptage",
    "Nombre de demandes approuvées",
    "Indique la capacité passée du client à obtenir un crédit."
),

"NB_REFUSED": (
    "PREVIOUS",
    "Comptage",
    "Nombre de demandes refusées",
    "Un nombre élevé de refus peut signaler un profil de risque plus élevé."
),

"NB_CANCELED": (
    "PREVIOUS",
    "Comptage",
    "Nombre de demandes annulées",
    "Décrit le comportement du client vis-à-vis de ses demandes précédentes."
),

"NB_UNUSED": (
    "PREVIOUS",
    "Comptage",
    "Nombre de crédits accordés mais non utilisés",
    "Permet de caractériser l'utilisation effective des financements obtenus."
),

"APPROVAL_RATE": (
    "PREVIOUS",
    "Ratio",
    "Taux d'approbation des demandes précédentes",
    "Mesure la proportion de demandes précédentes ayant été acceptées."
),

"REFUSAL_RATE": (
    "PREVIOUS",
    "Ratio",
    "Taux de refus des demandes précédentes",
    "Indicateur synthétique du niveau de difficulté rencontré lors des demandes précédentes."
),

# ---------------- POS CASH ----------------

"NB_POS_RECORDS": (
    "POS_CASH",
    "Comptage",
    "Nombre d'enregistrements POS Cash",
    "Mesure le volume d'historique disponible."
),

"NB_POS_CREDITS": (
    "POS_CASH",
    "Comptage",
    "Nombre de crédits POS Cash",
    "Indique le nombre de crédits concernés par cet historique."
),

"NB_ACTIVE_POS": (
    "POS_CASH",
    "Comptage",
    "Nombre de crédits POS actifs",
    "Mesure les engagements POS encore actifs."
),

"NB_COMPLETED_POS": (
    "POS_CASH",
    "Comptage",
    "Nombre de crédits POS terminés",
    "Indique le volume de crédits déjà remboursés ou clôturés."
),

"NB_DELAY_MONTHS_x": (
    "POS_CASH",
    "Comptage",
    "Nombre de mois présentant un retard",
    "Mesure la fréquence des retards dans l'historique POS."
),

"MAX_DPD_x": (
    "POS_CASH",
    "Maximum",
    "Nombre maximal de jours de retard",
    "Mesure le retard de paiement le plus important observé."
),

"MAX_DPD_DEF_x": (
    "POS_CASH",
    "Maximum",
    "Retard maximal associé aux situations de défaut",
    "Permet d'identifier les épisodes de retard les plus sévères."
),

"AVG_CNT_INSTALMENT": (
    "POS_CASH",
    "Moyenne",
    "Nombre moyen d'échéances",
    "Caractérise la structure moyenne des crédits POS."
),

"AVG_CNT_INSTALMENT_FUTURE": (
    "POS_CASH",
    "Moyenne",
    "Nombre moyen d'échéances futures",
    "Mesure les obligations de remboursement encore prévues."
),

"DPD_RATIO_x": (
    "POS_CASH",
    "Ratio",
    "Proportion de périodes présentant un retard",
    "Mesure la fréquence relative des retards."
),

"ACTIVE_RATIO": (
    "POS_CASH",
    "Ratio",
    "Proportion de crédits POS actifs",
    "Indique la part des engagements encore en cours."
),

# ---------------- CREDIT CARD ----------------

"NB_CREDIT_CARD_RECORDS": (
    "CREDIT_CARD",
    "Comptage",
    "Nombre d'enregistrements de cartes de crédit",
    "Mesure le volume d'historique disponible."
),

"NB_CREDIT_CARDS": (
    "CREDIT_CARD",
    "Comptage",
    "Nombre de cartes de crédit",
    "Mesure le nombre de cartes détenues ou suivies."
),

"AVG_BALANCE": (
    "CREDIT_CARD",
    "Moyenne",
    "Solde moyen des cartes",
    "Mesure le niveau moyen d'utilisation financière des cartes."
),

"MAX_BALANCE": (
    "CREDIT_CARD",
    "Maximum",
    "Solde maximal observé",
    "Identifie le niveau d'endettement maximal sur carte."
),

"AVG_CREDIT_LIMIT": (
    "CREDIT_CARD",
    "Moyenne",
    "Limite moyenne de crédit",
    "Mesure la capacité de crédit disponible."
),

"AVG_PAYMENT": (
    "CREDIT_CARD",
    "Moyenne",
    "Paiement moyen",
    "Caractérise le niveau moyen de remboursement sur carte."
),

"TOTAL_PAYMENT": (
    "CREDIT_CARD",
    "Somme",
    "Montant total payé",
    "Mesure l'ensemble des remboursements réalisés sur cartes."
),

"NB_DELAY_MONTHS_y": (
    "CREDIT_CARD",
    "Comptage",
    "Nombre de mois avec retard",
    "Indique la fréquence des retards de paiement."
),

"MAX_DPD_y": (
    "CREDIT_CARD",
    "Maximum",
    "Retard maximal en jours",
    "Mesure la gravité maximale des retards."
),

"MAX_DPD_DEF_y": (
    "CREDIT_CARD",
    "Maximum",
    "Retard maximal associé aux défauts",
    "Indicateur de sévérité des incidents de paiement."
),

"CREDIT_UTILIZATION_RATIO": (
    "CREDIT_CARD",
    "Ratio",
    "Ratio moyen d'utilisation du crédit disponible",
    "Une utilisation élevée peut indiquer une forte dépendance au crédit."
),

"DPD_RATIO_y": (
    "CREDIT_CARD",
    "Ratio",
    "Proportion de périodes avec retard",
    "Mesure la fréquence relative des retards sur cartes."
)
}


# ============================================================
# 4. INFORMATIONS SUR L'ENCODAGE
# ============================================================

encoding_info = {

"HOUSETYPE_MODE_Inconnu": (
    "HOUSETYPE_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si le type de logement est inconnu, sinon 0",
    "Permet au modèle de traiter une catégorie qualitative sans imposer d'ordre artificiel."
),

"HOUSETYPE_MODE_block of flats": (
    "HOUSETYPE_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si le logement est un appartement, sinon 0",
    "Transforme la catégorie en variable numérique exploitable par les modèles."
),

"HOUSETYPE_MODE_specific housing": (
    "HOUSETYPE_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si le logement appartient à cette catégorie, sinon 0",
    "Évite de créer une relation numérique artificielle entre les catégories."
),

"HOUSETYPE_MODE_terraced house": (
    "HOUSETYPE_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si le logement est une maison mitoyenne, sinon 0",
    "Permet de conserver l'information qualitative."
),

"WALLSMATERIAL_MODE_Block": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau = Block, sinon 0",
    "Conversion d'une catégorie qualitative en variable numérique."
),

"WALLSMATERIAL_MODE_Inconnu": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau inconnu, sinon 0",
    "Conserve explicitement l'information d'absence de connaissance."
),

"WALLSMATERIAL_MODE_Mixed": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau = Mixed, sinon 0",
    "Conversion en représentation numérique."
),

"WALLSMATERIAL_MODE_Monolithic": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau = Monolithic, sinon 0",
    "Conserve l'information sur le matériau."
),

"WALLSMATERIAL_MODE_Others": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau = Others, sinon 0",
    "Représentation numérique de la catégorie."
),

"WALLSMATERIAL_MODE_Panel": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau = Panel, sinon 0",
    "Représentation numérique de la catégorie."
),

"WALLSMATERIAL_MODE_Stone, brick": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau = Stone, brick, sinon 0",
    "Représentation numérique de la catégorie."
),

"WALLSMATERIAL_MODE_Wooden": (
    "WALLSMATERIAL_MODE",
    "One-Hot Encoding",
    "Variable catégorielle → indicatrice",
    "1 si matériau = Wooden, sinon 0",
    "Représentation numérique de la catégorie."
),

"age_tranche_20-30": (
    "age_tranche",
    "One-Hot Encoding",
    "Tranche d'âge → indicatrice",
    "1 si âge entre 20 et 30 ans, sinon 0",
    "Permet au modèle de capturer des effets non linéaires liés à l'âge."
),

"age_tranche_30-40": (
    "age_tranche",
    "One-Hot Encoding",
    "Tranche d'âge → indicatrice",
    "1 si âge entre 30 et 40 ans, sinon 0",
    "Permet de conserver l'information de la tranche d'âge."
),

"age_tranche_40-50": (
    "age_tranche",
    "One-Hot Encoding",
    "Tranche d'âge → indicatrice",
    "1 si âge entre 40 et 50 ans, sinon 0",
    "Permet au modèle de distinguer les profils d'âge."
),

"age_tranche_50-60": (
    "age_tranche",
    "One-Hot Encoding",
    "Tranche d'âge → indicatrice",
    "1 si âge entre 50 et 60 ans, sinon 0",
    "Permet de modéliser l'effet de l'âge."
),

"age_tranche_60-70": (
    "age_tranche",
    "One-Hot Encoding",
    "Tranche d'âge → indicatrice",
    "1 si âge entre 60 et 70 ans, sinon 0",
    "Permet de représenter la tranche d'âge sous forme numérique."
)
}


# ============================================================
# 5. CONSTRUCTION DU TABLEAU FINAL
# ============================================================

rows = []

# Features agrégées
for feature in aggregated_features:

    if feature in feature_info:
        source, transformation, description, justification = feature_info[feature]

        rows.append({
            "N°": len(rows) + 1,
            "Feature finale": feature,
            "Catégorie": "Feature agrégée",
            "Source": source,
            "Transformation": transformation,
            "Ce que représente la variable": description,
            "Justification métier": justification
        })

# Features encodées
for feature in encoded_features:

    source, transformation, description, valeur, justification = encoding_info[feature]

    rows.append({
        "N°": len(rows) + 1,
        "Feature finale": feature,
        "Catégorie": "Feature encodée",
        "Source": source,
        "Transformation": transformation,
        "Ce que représente la variable": valeur,
        "Justification métier": justification
    })


df_features = pd.DataFrame(rows)


# ============================================================
# 6. VÉRIFICATION
# ============================================================

print("==========================================")
print("DICTIONNAIRE DES FEATURES")
print("==========================================")

print("Nombre total de features :", len(df_features))

print("\nFeatures agrégées :",
      (df_features["Catégorie"] == "Feature agrégée").sum())

print("Features encodées :",
      (df_features["Catégorie"] == "Feature encodée").sum())


# ============================================================
# 7. CRÉATION DU DOSSIER
# ============================================================

os.makedirs("data/processed", exist_ok=True)


# ============================================================
# 8. EXPORT EXCEL
# ============================================================

output_file = "data/processed/dictionnaire_features.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    df_features.to_excel(
        writer,
        sheet_name="Dictionnaire Features",
        index=False
    )

    # Feuille séparée : résumé
    resume = pd.DataFrame({
        "Élément": [
            "Features de base avant agrégation",
            "Features agrégées",
            "Features créées par encodage",
            "Total des features ajoutées",
            "Nombre final de colonnes TRAIN",
            "Nombre final de colonnes TEST"
        ],
        "Nombre": [
            165,
            63,
            17,
            80,
            228,
            228
        ]
    })

    resume.to_excel(
        writer,
        sheet_name="Résumé",
        index=False
    )


# ============================================================
# 9. MISE EN FORME EXCEL
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb = load_workbook(output_file)

# -------------------------------
# Feuille dictionnaire
# -------------------------------

ws = wb["Dictionnaire Features"]

# En-têtes
for cell in ws[1]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7"
    )
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )

# Largeur des colonnes
widths = {
    "A": 8,
    "B": 40,
    "C": 22,
    "D": 22,
    "E": 25,
    "F": 65,
    "G": 75
}

for col, width in widths.items():
    ws.column_dimensions[col].width = width

# Retour à la ligne
for row in ws.iter_rows():
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )

# Figer la première ligne
ws.freeze_panes = "A2"

# Filtre automatique
ws.auto_filter.ref = ws.dimensions


# -------------------------------
# Feuille résumé
# -------------------------------

ws2 = wb["Résumé"]

for cell in ws2[1]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7"
    )

ws2.column_dimensions["A"].width = 45
ws2.column_dimensions["B"].width = 20

ws2.freeze_panes = "A2"


# ============================================================
# 10. SAUVEGARDE
# ============================================================

wb.save(output_file)

print("\n==========================================")
print("FICHIER EXCEL CRÉÉ")
print("==========================================")

print("Fichier :", output_file)
print("Nombre de lignes :", len(df_features))

DICTIONNAIRE DES FEATURES
Nombre total de features : 80

Features agrégées : 63
Features encodées : 17

FICHIER EXCEL CRÉÉ
Fichier : data/processed/dictionnaire_features.xlsx
Nombre de lignes : 80


In [40]:
# ============================================================
# VÉRIFICATION DES DATASETS ENCODÉS
# ============================================================

print("TRAIN encoded :", train_encoded.shape)
print("TEST encoded  :", test_encoded.shape)

print("\nVariables non numériques TRAIN :")
print(
    train_encoded.select_dtypes(exclude=np.number).columns.tolist()
)

print("\nVariables non numériques TEST :")
print(
    test_encoded.select_dtypes(exclude=np.number).columns.tolist()
)

TRAIN encoded : (307511, 242)
TEST encoded  : (48744, 242)

Variables non numériques TRAIN :
[]

Variables non numériques TEST :
[]


In [41]:
train_encoded.to_csv(
    "data/processed/train_encoded.csv",
    index=False
)

test_encoded.to_csv(
    "data/processed/test_encoded.csv",
    index=False
)

print("TRAIN encoded sauvegardé :", train_encoded.shape)
print("TEST encoded sauvegardé  :", test_encoded.shape)

TRAIN encoded sauvegardé : (307511, 242)
TEST encoded sauvegardé  : (48744, 242)
